
# DualROIUNet: Explainable Multimodal MRI–FDG-PET Alzheimer's Diagnosis

**Notebook implementation based on the uploaded manuscript.**  
Primary task: subject-level **CN vs AD** classification with joint ROI segmentation. MCI is kept outside the primary cross-validation and can be evaluated separately.

## Dataset contract (prepare this before training)

Create a CSV manifest with one row per participant:

| subject_id | diagnosis | mri_path | pet_path | mask_path |
|---|---|---|---|---|
| sub001 | CN | `/.../mri.npy` | `/.../pet.npy` | `/.../roi.npy` |
| sub002 | AD | `/.../mri.npy` | `/.../pet.npy` | `/.../roi.npy` |

Each `.npy` file must contain a 3-D array in **[slices, height, width]** order. MRI, PET and mask arrays for a participant must be co-registered and have the same slice count and spatial coordinates. Masks are binary ROI masks derived from the AAL atlas registration described in the manuscript, covering the hippocampus, entorhinal cortex, posterior cingulate cortex and precuneus.

If your data are NIfTI (`.nii`/`.nii.gz`), the conversion helper below loads them with nibabel and extracts axial slices. Skull stripping, MRI–PET registration, spatial normalization and AAL atlas registration should be performed with a validated neuroimaging workflow (e.g. your ADNI/SPM12 preprocessing outputs) before training. This notebook does not fabricate anatomical masks.

**Important:** the reported manuscript metrics (99.48% accuracy, Dice 0.985) are manuscript-reported results, not guaranteed outputs. Actual results depend on the data, preprocessing, split, and hardware.


In [ ]:

# 1. Install dependencies (Google Colab)
!pip -q install nibabel scikit-learn scikit-image lime pandas matplotlib seaborn tqdm


In [ ]:

# 2. Imports and reproducibility
import os, random, copy, json, warnings, math, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef, roc_curve, classification_report)
from sklearn.preprocessing import MinMaxScaler

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


In [ ]:

# 3. Configuration — EDIT THESE PATHS
# In Colab, mount Google Drive first if your data are stored there.
from google.colab import drive
drive.mount('/content/drive')

MANIFEST_CSV = "/content/drive/MyDrive/ADNI_DualROIUNet/manifest.csv"
OUTPUT_DIR = "/content/drive/MyDrive/ADNI_DualROIUNet/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMAGE_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 100
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2
N_SPLITS = 10
PATIENCE = 15
BASE_CHANNELS = 16   # Increase to 32 for larger GPUs; architecture remains the same.
MAX_SLICES_PER_SUBJECT = 40


In [ ]:

# 4. Optional NIfTI-to-NPY conversion utility
# Run only if your source volumes are NIfTI. Input volumes should already be co-registered.
def nifti_to_axial_npy(nifti_path, output_npy, target_size=256, central_fraction=0.60,
                       max_slices=40, is_mask=False):
    import nibabel as nib
    from scipy.ndimage import zoom, median_filter
    vol = nib.load(str(nifti_path)).get_fdata().astype(np.float32)
    # NIfTI arrays are commonly X,Y,Z. Move axial Z to first dimension.
    vol = np.transpose(vol, (2, 1, 0))
    n = vol.shape[0]
    lo = int((1-central_fraction)/2*n); hi = int((1+central_fraction)/2*n)
    idx = np.linspace(lo, max(lo, hi-1), min(max_slices, max(1, hi-lo))).astype(int)
    slices = []
    for z in idx:
        sl = vol[z]
        factors = (target_size/sl.shape[0], target_size/sl.shape[1])
        sl = zoom(sl, factors, order=0 if is_mask else 3)
        if is_mask:
            sl = (sl > 0).astype(np.float32)
        else:
            sl = median_filter(sl, size=3)
            sd = sl.std()
            sl = (sl-sl.mean())/(sd+1e-6)
        slices.append(sl.astype(np.float32))
    arr = np.stack(slices)
    np.save(output_npy, arr)
    return arr.shape

# Example:
# nifti_to_axial_npy("sub001_T1.nii.gz", "sub001_mri.npy")
# nifti_to_axial_npy("sub001_FDG.nii.gz", "sub001_pet.npy")
# nifti_to_axial_npy("sub001_AAL_ROI.nii.gz", "sub001_roi.npy", is_mask=True)


In [ ]:

# 5. Manifest validation and image preprocessing
required_cols = {"subject_id", "diagnosis", "mri_path", "pet_path", "mask_path"}
manifest = pd.read_csv(MANIFEST_CSV)
missing = required_cols - set(manifest.columns)
if missing:
    raise ValueError(f"Manifest is missing columns: {sorted(missing)}")

manifest["diagnosis"] = manifest["diagnosis"].astype(str).str.upper().str.strip()
manifest = manifest[manifest["diagnosis"].isin(["CN", "AD", "MCI"])].copy()
for col in ["mri_path", "pet_path", "mask_path"]:
    manifest[col] = manifest[col].astype(str)
    manifest = manifest[manifest[col].map(os.path.exists)].copy()

# Primary experiment excludes MCI, as specified by the manuscript.
primary = manifest[manifest.diagnosis.isin(["CN", "AD"])].copy().reset_index(drop=True)
primary["label"] = primary.diagnosis.map({"CN": 0, "AD": 1}).astype(int)

if primary.subject_id.duplicated().any():
    raise ValueError("Manifest must have exactly one row per subject_id.")
if len(primary) < N_SPLITS:
    raise ValueError(f"Need at least {N_SPLITS} CN/AD subjects for {N_SPLITS}-fold CV.")

print("Primary CN/AD participants:", len(primary))
print(primary.diagnosis.value_counts())
print("MCI participants held out for exploratory analysis:", int((manifest.diagnosis=="MCI").sum()))

def load_array(path):
    path = str(path)
    if path.endswith(".npz"):
        z = np.load(path)
        return z[z.files[0]].astype(np.float32)
    return np.load(path).astype(np.float32)

def preprocess_slice(x, is_mask=False):
    from scipy.ndimage import median_filter
    x = np.asarray(x, dtype=np.float32)
    if x.ndim != 2: raise ValueError(f"Expected 2D slice, got {x.shape}")
    # Resize to 256x256. Bicubic for images, nearest-neighbour for masks.
    from skimage.transform import resize
    x = resize(x, (IMAGE_SIZE, IMAGE_SIZE), order=0 if is_mask else 3,
               preserve_range=True, anti_aliasing=not is_mask).astype(np.float32)
    if is_mask:
        return (x > 0.5).astype(np.float32)
    x = median_filter(x, size=3)
    x = (x - x.mean()) / (x.std() + 1e-6)
    return x.astype(np.float32)

def load_subject_arrays(row):
    mri, pet, mask = [load_array(row[c]) for c in ["mri_path","pet_path","mask_path"]]
    if mri.ndim == 2: mri = mri[None]
    if pet.ndim == 2: pet = pet[None]
    if mask.ndim == 2: mask = mask[None]
    if not (mri.shape[0] == pet.shape[0] == mask.shape[0]):
        raise ValueError(f"Slice-count mismatch for subject {row.subject_id}: {mri.shape}, {pet.shape}, {mask.shape}")
    n = mri.shape[0]
    # Keep the central 60% and sample up to 40 consecutive/equally spaced slices.
    lo, hi = int(0.2*n), max(int(0.8*n), int(0.2*n)+1)
    ids = np.linspace(lo, hi-1, min(MAX_SLICES_PER_SUBJECT, hi-lo)).astype(int)
    xs, ys = [], []
    for i in ids:
        a = preprocess_slice(mri[i]); b = preprocess_slice(pet[i])
        m = preprocess_slice(mask[i], is_mask=True)
        # Retain slices with ROI content; if masks are empty, preserve the slice so errors are visible.
        if m.sum() > 0:
            xs.append(np.stack([a,b], axis=0)); ys.append(m[None])
    if not xs:
        raise ValueError(f"No non-empty ROI slices for {row.subject_id}. Check AAL masks.")
    return np.stack(xs), np.stack(ys)


In [ ]:

# 6. Subject-aware slice dataset
class SubjectSliceDataset(Dataset):
    def __init__(self, frame):
        self.images, self.masks, self.labels, self.subject_ids = [], [], [], []
        for row in frame.itertuples(index=False):
            x, y = load_subject_arrays(row)
            self.images.append(torch.from_numpy(x).float())
            self.masks.append(torch.from_numpy(y).float())
            self.labels.append(int(row.label))
            self.subject_ids.append(str(row.subject_id))
        self.offsets = np.cumsum([0] + [len(x) for x in self.images])
    def __len__(self): return int(self.offsets[-1])
    def __getitem__(self, idx):
        s = int(np.searchsorted(self.offsets[1:], idx, side="right"))
        local = idx - self.offsets[s]
        return self.images[s][local], self.masks[s][local], torch.tensor(self.labels[s]), s

def make_loaders(train_df, val_df, test_df):
    train_ds, val_ds, test_ds = map(SubjectSliceDataset, [train_df, val_df, test_df])
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader


## 7. DualROIUNet architecture
The MRI branch uses a differentiable Haar-style wavelet decomposition (LL/LH/HL/HH) followed by residual-dense blocks and pyramidal fusion. The PET branch uses 7×7 depthwise dilated convolutions and parameter-free average/max channel gating. Bi-directional channel/spatial attention fuses both modalities. A dual-skip decoder predicts ROI masks, and ROI-GateNet performs mask-guided classification.

In [ ]:

# 7A. Building blocks
class ConvBNReLU(nn.Module):
    def __init__(self, cin, cout, k=3, stride=1, dilation=1, groups=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, k, stride, padding=(k//2)*dilation,
                      dilation=dilation, groups=groups, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class ResidualDenseBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c1 = ConvBNReLU(c, c); self.c2 = ConvBNReLU(2*c, c)
        self.c3 = ConvBNReLU(3*c, c)
        self.reduce = nn.Conv2d(4*c, c, 1, bias=False)
        self.bn = nn.BatchNorm2d(c)
    def forward(self, x):
        a = self.c1(x); b = self.c2(torch.cat([x,a],1))
        c = self.c3(torch.cat([x,a,b],1))
        return F.relu(self.bn(self.reduce(torch.cat([x,a,b,c],1))) + x)

def haar_wavelet(x):
    # Differentiable 2x2 Haar sub-bands, each at half resolution.
    a = x[:,:,0::2,0::2]; b = x[:,:,0::2,1::2]
    c = x[:,:,1::2,0::2]; d = x[:,:,1::2,1::2]
    ll=(a+b+c+d)*0.5; lh=(-a-b+c+d)*0.5
    hl=(-a+b-c+d)*0.5; hh=(a-b-c+d)*0.5
    return ll, lh, hl, hh

class PWResNetEncoder(nn.Module):
    def __init__(self, in_ch=1, base=16):
        super().__init__()
        self.stem = ConvBNReLU(in_ch, base)
        self.rd = ResidualDenseBlock(base)
        self.band_fuse = nn.Sequential(ConvBNReLU(base*4, base), ConvBNReLU(base,base,dilation=2))
        self.down = nn.ModuleList([ConvBNReLU(base,base*2,3,2),
                                   ConvBNReLU(base*2,base*4,3,2),
                                   ConvBNReLU(base*4,base*8,3,2)])
        self.blocks = nn.ModuleList([ResidualDenseBlock(base*2),
                                     ResidualDenseBlock(base*4),
                                     ResidualDenseBlock(base*8)])
    def forward(self,x):
        x=self.stem(x)
        bands=haar_wavelet(x)
        bands=[self.rd(z) for z in bands]
        x=self.band_fuse(torch.cat(bands,1))
        skips=[x]
        for down, block in zip(self.down,self.blocks):
            x=block(down(x)); skips.append(x)
        return skips

class StatisticalChannelGate(nn.Module):
    def forward(self,x):
        avg=x.mean((2,3),keepdim=True)
        mx=x.amax((2,3),keepdim=True)
        return x*torch.sigmoid(avg+mx)

class DDLFCBlock(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.dw=nn.Conv2d(c,c,7,padding=6,dilation=2,groups=c,bias=False)
        self.pw=ConvBNReLU(c,c,1)
        self.gate=StatisticalChannelGate()
        self.refine=ConvBNReLU(c,c)
    def forward(self,x): return self.refine(self.gate(self.pw(self.dw(x)))+x)

class CADDLFCNetEncoder(nn.Module):
    def __init__(self,in_ch=1,base=16):
        super().__init__()
        widths=[base,base*2,base*4,base*8]
        self.stem=ConvBNReLU(in_ch,widths[0])
        self.blocks=nn.ModuleList([DDLFCBlock(c) for c in widths])
        self.down=nn.ModuleList([ConvBNReLU(widths[i],widths[i+1],3,2) for i in range(3)])
    def forward(self,x):
        x=self.stem(x); skips=[]
        for i,block in enumerate(self.blocks):
            x=block(x); skips.append(x)
            if i < len(self.down): x=self.down[i](x)
        return skips

class BiDirectionalAttentionFusion(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.mri_channel=nn.Conv2d(c,c,1); self.pet_channel=nn.Conv2d(c,c,1)
        self.mri_spatial=nn.Conv2d(2,1,7,padding=3)
        self.pet_spatial=nn.Conv2d(2,1,7,padding=3)
        self.compress=ConvBNReLU(2*c,c,1)
    def spatial(self,x,conv):
        return torch.sigmoid(conv(torch.cat([x.mean(1,keepdim=True),x.amax(1,keepdim=True)],1)))
    def forward(self,m,p):
        # Opposite modality provides channel attention; each modality supplies spatial context.
        m_att=m*torch.sigmoid(self.pet_channel(F.adaptive_avg_pool2d(p,1)))
        p_att=p*torch.sigmoid(self.mri_channel(F.adaptive_avg_pool2d(m,1)))
        m_att=m_att*self.spatial(p,self.mri_spatial)
        p_att=p_att*self.spatial(m,self.pet_spatial)
        return self.compress(torch.cat([m_att,p_att],1))


In [ ]:

# 7B. Dual-skip segmentation decoder + ROI-GateNet
class DecoderBlock(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.refine=nn.Sequential(ConvBNReLU(cin+2*cskip,cout),
                                  ConvBNReLU(cout,cout))
    def forward(self,x,ms,ps):
        x=F.interpolate(x,size=ms.shape[-2:],mode="bilinear",align_corners=False)
        return self.refine(torch.cat([x,ms,ps],1))

class DualROIUNet(nn.Module):
    def __init__(self,base=16,num_classes=2):
        super().__init__()
        self.mri_encoder=PWResNetEncoder(1,base)
        self.pet_encoder=CADDLFCNetEncoder(1,base)
        widths=[base,base*2,base*4,base*8]
        self.fusions=nn.ModuleList([BiDirectionalAttentionFusion(c) for c in widths])
        self.bottleneck=nn.Sequential(ConvBNReLU(widths[-1],widths[-1]),
                                      ConvBNReLU(widths[-1],widths[-1]))
        self.dec3=DecoderBlock(widths[3],widths[2],widths[2])
        self.dec2=DecoderBlock(widths[2],widths[1],widths[1])
        self.dec1=DecoderBlock(widths[1],widths[0],widths[0])
        self.mask_head=nn.Conv2d(widths[0],1,1)
        self.classifier=nn.Sequential(nn.Linear(widths[0],128),nn.ReLU(),
                                      nn.Dropout(0.3),nn.Linear(128,num_classes))
        self.last_attention=None
    def forward(self,x):
        mri=x[:,0:1]; pet=x[:,1:2]
        ms=self.mri_encoder(mri); ps=self.pet_encoder(pet)
        fused=[f(a,b) for f,a,b in zip(self.fusions,ms,ps)]
        z=self.bottleneck(fused[-1])
        d=self.dec3(z,ms[2],ps[2])
        d=self.dec2(d,ms[1],ps[1])
        d=self.dec1(d,ms[0],ps[0])
        mask_logits=self.mask_head(d)
        mask=torch.sigmoid(mask_logits)
        gated=d*mask
        pooled=F.adaptive_avg_pool2d(gated,1).flatten(1)
        class_logits=self.classifier(pooled)
        self.last_attention=mask.detach()
        return class_logits, mask_logits

model=DualROIUNet(base=BASE_CHANNELS).to(DEVICE)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")


In [ ]:

# 8. Loss functions and segmentation metrics
def dice_loss_from_logits(logits, target, eps=1e-6):
    prob=torch.sigmoid(logits)
    dims=(1,2,3)
    inter=(prob*target).sum(dims)
    denom=prob.sum(dims)+target.sum(dims)
    dice=(2*inter+eps)/(denom+eps)
    return 1-dice.mean()

def segmentation_loss(logits,target):
    bce=F.binary_cross_entropy_with_logits(logits,target)
    dice=dice_loss_from_logits(logits,target)
    return 0.7*dice+0.3*bce

@torch.no_grad()
def segmentation_scores(logits,target,threshold=0.5):
    pred=(torch.sigmoid(logits)>threshold).float()
    dims=(1,2,3)
    inter=(pred*target).sum(dims)
    union=((pred+target)>0).float().sum(dims)
    dice=(2*inter+1e-6)/(pred.sum(dims)+target.sum(dims)+1e-6)
    iou=(inter+1e-6)/(union+1e-6)
    pix=(pred==target).float().mean((1,2,3))
    return dice.mean().item(),iou.mean().item(),pix.mean().item()

def combined_loss(class_logits, labels, mask_logits, masks):
    return F.cross_entropy(class_logits,labels)+segmentation_loss(mask_logits,masks)


In [ ]:

# 9. Training and participant-level aggregation
def run_epoch(model,loader,optimizer=None):
    train=optimizer is not None
    model.train(train)
    total_loss=0; n=0
    all_labels=[]; all_probs=[]; all_preds=[]; all_dice=[]; all_iou=[]
    subject_slice_probs={}
    subject_labels={}
    for x,m,y,sidx in loader:
        x=x.to(DEVICE); m=m.to(DEVICE); y=y.to(DEVICE)
        with torch.set_grad_enabled(train):
            logits,mask_logits=model(x)
            loss=combined_loss(logits,y,mask_logits,m)
            if train:
                optimizer.zero_grad(set_to_none=True); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
                optimizer.step()
        bs=len(y); total_loss+=loss.item()*bs; n+=bs
        probs=torch.softmax(logits,1)[:,1].detach().cpu().numpy()
        preds=(probs>=0.5).astype(int)
        all_labels.extend(y.detach().cpu().numpy().tolist())
        all_probs.extend(probs.tolist()); all_preds.extend(preds.tolist())
        d,i,_=segmentation_scores(mask_logits.detach(),m)
        all_dice.append(d); all_iou.append(i)
        for j,si in enumerate(sidx.numpy().tolist()):
            subject_slice_probs.setdefault(int(si),[]).append(float(probs[j]))
            subject_labels[int(si)]=int(y[j].item())
    return {"loss":total_loss/max(n,1),"slice_labels":np.array(all_labels),
            "slice_probs":np.array(all_probs),"slice_preds":np.array(all_preds),
            "dice":float(np.mean(all_dice)) if all_dice else np.nan,
            "iou":float(np.mean(all_iou)) if all_iou else np.nan,
            "subject_probs":{k:float(np.mean(v)) for k,v in subject_slice_probs.items()},
            "subject_labels":subject_labels}

def subject_metrics(result):
    keys=sorted(result["subject_probs"])
    y=np.array([result["subject_labels"][k] for k in keys])
    p=np.array([result["subject_probs"][k] for k in keys])
    pred=(p>=0.5).astype(int)
    out={"accuracy":accuracy_score(y,pred),
         "precision":precision_score(y,pred,zero_division=0),
         "sensitivity":recall_score(y,pred,zero_division=0),
         "specificity":recall_score(y,pred,pos_label=0,zero_division=0),
         "f1":f1_score(y,pred,zero_division=0),
         "mcc":matthews_corrcoef(y,pred),
         "auc":roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan,
         "y_true":y,"y_prob":p,"y_pred":pred,"subject_keys":keys}
    return out


In [ ]:

# 10. Stratified 10-fold subject-level CV
# The split occurs at participant level BEFORE any slices are loaded into a fold.
skf=StratifiedKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED)
fold_rows=[]; all_oof=[]; fold_histories=[]
for fold,(trainval_idx,test_idx) in enumerate(skf.split(primary,primary.label),start=1):
    print(f"\\n========== FOLD {fold}/{N_SPLITS} ==========")
    trainval_df=primary.iloc[trainval_idx].reset_index(drop=True)
    test_df=primary.iloc[test_idx].reset_index(drop=True)
    train_df,val_df=train_test_split(trainval_df,test_size=0.1111,
        stratify=trainval_df.label,random_state=SEED+fold)
    train_df=train_df.reset_index(drop=True); val_df=val_df.reset_index(drop=True)
    train_ds,val_ds,test_ds,train_loader,val_loader,test_loader=make_loaders(train_df,val_df,test_df)

    net=DualROIUNet(base=BASE_CHANNELS).to(DEVICE)
    optimizer=torch.optim.Adam(net.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="min",factor=0.5,patience=5)
    best_val=float("inf"); best_state=None; wait=0
    history={"train_loss":[],"val_loss":[],"val_dice":[]}
    for epoch in range(1,EPOCHS+1):
        tr=run_epoch(net,train_loader,optimizer)
        va=run_epoch(net,val_loader)
        scheduler.step(va["loss"])
        history["train_loss"].append(tr["loss"]); history["val_loss"].append(va["loss"])
        history["val_dice"].append(va["dice"])
        if va["loss"] < best_val:
            best_val=va["loss"]; best_state=copy.deepcopy(net.state_dict()); wait=0
        else:
            wait+=1
        if epoch==1 or epoch%5==0:
            print(f"Epoch {epoch:03d}/{EPOCHS} | train {tr['loss']:.4f} | val {va['loss']:.4f} | val Dice {va['dice']:.4f}")
        if wait>=PATIENCE:
            print("Early stopping."); break
    net.load_state_dict(best_state)
    te=run_epoch(net,test_loader)
    sm=subject_metrics(te)
    row={k:sm[k] for k in ["accuracy","precision","sensitivity","specificity","f1","mcc","auc"]}
    row.update({"fold":fold,"seg_dice":te["dice"],"seg_iou":te["iou"],
                "test_subjects":len(test_df),"best_val_loss":best_val})
    fold_rows.append(row); fold_histories.append(history)
    for sid,y,pred,prob in zip(sm["subject_keys"],sm["y_true"],sm["y_pred"],sm["y_prob"]):
        all_oof.append({"fold":fold,"subject_id":test_ds.subject_ids[sid],
                        "y_true":int(y),"y_pred":int(pred),"AD_probability":float(prob)})
    torch.save(net.state_dict(),os.path.join(OUTPUT_DIR,f"DualROIUNet_fold{fold}.pth"))
    del net,optimizer,train_ds,val_ds,test_ds,train_loader,val_loader,test_loader
    if torch.cuda.is_available(): torch.cuda.empty_cache()

fold_df=pd.DataFrame(fold_rows)
oof_df=pd.DataFrame(all_oof)
fold_df.to_csv(os.path.join(OUTPUT_DIR,"fold_metrics.csv"),index=False)
oof_df.to_csv(os.path.join(OUTPUT_DIR,"subject_level_predictions.csv"),index=False)
display(fold_df)


In [ ]:

# 11. Cross-validation summary and plots
metric_cols=["accuracy","precision","sensitivity","specificity","f1","mcc","auc","seg_dice","seg_iou"]
summary=pd.DataFrame({"Metric":metric_cols,
    "Mean": [fold_df[c].mean() for c in metric_cols],
    "Std": [fold_df[c].std(ddof=1) for c in metric_cols]})
display(summary)
summary.to_csv(os.path.join(OUTPUT_DIR,"cv_summary.csv"),index=False)

plt.figure(figsize=(10,5))
plt.plot(fold_df.fold,fold_df.accuracy,marker="o",label="Accuracy")
plt.plot(fold_df.fold,fold_df.f1,marker="s",label="F1")
plt.plot(fold_df.fold,fold_df.auc,marker="^",label="ROC-AUC")
plt.xlabel("Fold"); plt.ylabel("Score"); plt.ylim(0,1.02); plt.title("Subject-level 10-fold performance")
plt.grid(alpha=.3); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"fold_performance.png"),dpi=300); plt.show()

plt.figure(figsize=(9,5))
sns.barplot(data=summary[summary.Metric.isin(metric_cols[:7])],x="Metric",y="Mean")
plt.ylim(0,1.02); plt.xticks(rotation=30); plt.title("Mean classification metrics")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,"classification_metrics.png"),dpi=300); plt.show()

# Pooled out-of-fold confusion matrix and ROC curve
cm=confusion_matrix(oof_df.y_true,oof_df.y_pred)
plt.figure(figsize=(5,4)); sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",
    xticklabels=["CN","AD"],yticklabels=["CN","AD"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Pooled subject-level confusion matrix")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,"confusion_matrix.png"),dpi=300); plt.show()

fpr,tpr,_=roc_curve(oof_df.y_true,oof_df.AD_probability)
auc=roc_auc_score(oof_df.y_true,oof_df.AD_probability)
plt.figure(figsize=(5,5)); plt.plot(fpr,tpr,label=f"DualROIUNet AUC={auc:.4f}")
plt.plot([0,1],[0,1],"--"); plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Subject-level ROC curve"); plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"roc_curve.png"),dpi=300); plt.show()



## 12. Explainability: model attention and LIME

The decoder's predicted ROI probability map is the model-derived spatial attention/ROI output. LIME below explains the classifier's local decision from the two-channel input. LIME is run on a representative held-out subject slice; the explanation is not a clinical validation.


In [ ]:

# 12A. Load one saved fold model and visualize MRI, PET, ROI mask and attention.
# Uses the final fold checkpoint as a convenient demonstration; use the relevant fold checkpoint
# for a specific held-out participant to avoid evaluating that participant with a model trained on it.
demo_model=DualROIUNet(base=BASE_CHANNELS).to(DEVICE)
demo_ckpt=os.path.join(OUTPUT_DIR,f"DualROIUNet_fold{N_SPLITS}.pth")
demo_model.load_state_dict(torch.load(demo_ckpt,map_location=DEVICE))
demo_model.eval()

demo_row=primary.iloc[0]
demo_x,demo_y=load_subject_arrays(demo_row)
sample=torch.from_numpy(demo_x[0:1]).float().to(DEVICE)
with torch.no_grad():
    cls_logits,mask_logits=demo_model(sample)
    attn=torch.sigmoid(mask_logits)[0,0].cpu().numpy()
    prob=torch.softmax(cls_logits,1)[0,1].item()

fig,ax=plt.subplots(1,4,figsize=(16,4))
ax[0].imshow(demo_x[0,0],cmap="gray"); ax[0].set_title("MRI")
ax[1].imshow(demo_x[0,1],cmap="hot"); ax[1].set_title("FDG-PET")
ax[2].imshow(demo_y[0,0],cmap="gray"); ax[2].set_title("AAL ROI ground truth")
ax[3].imshow(demo_x[0,0],cmap="gray"); ax[3].imshow(attn,cmap="jet",alpha=.45); ax[3].set_title(f"ROI attention | AD p={prob:.3f}")
for a in ax: a.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,"attention_visualization.png"),dpi=300); plt.show()


In [ ]:

# 12B. LIME explanation for the two-modality slice
from lime import lime_image
from skimage.segmentation import mark_boundaries

def lime_predict(rgb_batch):
    # LIME expects HxWx3 images. Encode MRI, PET and their mean as the 3 channels.
    arr=np.asarray(rgb_batch,dtype=np.float32)
    x=np.stack([arr[...,0],arr[...,1]],axis=1)  # N,2,H,W
    xt=torch.tensor(x,dtype=torch.float32,device=DEVICE)
    with torch.no_grad():
        logits,_=demo_model(xt)
        return torch.softmax(logits,1).cpu().numpy()

rgb=np.stack([demo_x[0,0],demo_x[0,1],(demo_x[0,0]+demo_x[0,1])/2],axis=-1)
explainer=lime_image.LimeImageExplainer()
explanation=explainer.explain_instance(rgb, lime_predict, top_labels=2,
    hide_color=0, num_samples=1000, segmentation_fn=None)
label_to_show=int(np.argmax(lime_predict(rgb[None])[0]))
lime_img, lime_mask=explanation.get_image_and_mask(label_to_show,
    positive_only=False,num_features=10,hide_rest=False)
plt.figure(figsize=(6,6)); plt.imshow(mark_boundaries(lime_img,lime_mask))
plt.title("LIME: top 10 superpixels"); plt.axis("off"); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"lime_explanation.png"),dpi=300); plt.show()



## 13. Exploratory MCI analysis (not part of primary training)

MCI participants were excluded from CN-vs-AD training and all primary 10-fold test folds. This section can report model scores for MCI only after you choose and document an evaluation protocol. The code below performs inference with the final-fold model as an exploratory demonstration only; it is **not** a validated MCI classifier and does not estimate MCI-to-AD conversion.


In [ ]:

# 13. Optional MCI inference demonstration
mci_df=manifest[manifest.diagnosis=="MCI"].copy()
if len(mci_df):
    mci_df["label"]=0  # placeholder only to satisfy dataset class; do NOT interpret as ground truth.
    mci_ds=SubjectSliceDataset(mci_df)
    mci_loader=DataLoader(mci_ds,batch_size=BATCH_SIZE,shuffle=False)
    mci_model=demo_model
    mci_model.eval()
    probs_by_subject={}
    with torch.no_grad():
        for x,m,y,sidx in mci_loader:
            logits,_=mci_model(x.to(DEVICE))
            probs=torch.softmax(logits,1)[:,1].cpu().numpy()
            for si,p in zip(sidx.numpy().tolist(),probs):
                probs_by_subject.setdefault(int(si),[]).append(float(p))
    mci_scores=pd.DataFrame([{"subject_id":mci_ds.subject_ids[i],
                              "mean_AD_probability":float(np.mean(probs_by_subject.get(i,[np.nan])))}
                             for i in range(len(mci_ds.subject_ids))])
    mci_scores.to_csv(os.path.join(OUTPUT_DIR,"MCI_exploratory_scores_NOT_VALIDATED.csv"),index=False)
    display(mci_scores.head())
else:
    print("No MCI participants found in the manifest.")


In [ ]:

# 14. Export all results to Excel and ZIP
import zipfile
excel_path=os.path.join(OUTPUT_DIR,"DualROIUNet_Results.xlsx")
with pd.ExcelWriter(excel_path,engine="openpyxl") as writer:
    fold_df.to_excel(writer,sheet_name="Fold_Metrics",index=False)
    summary.to_excel(writer,sheet_name="CV_Summary",index=False)
    oof_df.to_excel(writer,sheet_name="Subject_Predictions",index=False)
    if "mci_scores" in globals():
        mci_scores.to_excel(writer,sheet_name="MCI_Exploratory",index=False)

zip_path=os.path.join(OUTPUT_DIR,"DualROIUNet_Complete_Outputs.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for root,dirs,files in os.walk(OUTPUT_DIR):
        for file in files:
            if file != os.path.basename(zip_path):
                full=os.path.join(root,file)
                z.write(full,arcname=os.path.relpath(full,OUTPUT_DIR))
print("Saved Excel:",excel_path)
print("Saved ZIP:",zip_path)



## 15. Implementation notes and reproducibility checks

- The manuscript specifies subject-level stratified 10-fold CV, with slices extracted only after subject splitting; this notebook follows that order.
- Training settings follow the manuscript: Adam, learning rate `1e-4`, weight decay `1e-5`, batch size `16`, up to `100` epochs, early stopping patience `15`, and segmentation loss `0.7 × Dice + 0.3 × BCE`.
- The model is a practical PyTorch implementation of the named modules. The paper does not provide source code or all low-level tensor equations; some implementation details (e.g. exact wavelet filter implementation and attention projections) are therefore explicit engineering realizations of the described operations.
- The notebook cannot recreate the reported performance without the same subject cohort, exact preprocessing, ROI atlas masks, and evaluation setup. Do not hard-code the manuscript's reported accuracy as an experimental result.
- For rigorous reporting, save the exact manifest, preprocessing versions, fold subject IDs, random seed, package versions, and model checkpoints.
- This is research code, not a medical device or a substitute for clinical diagnosis. Independent external and prospective validation are required.
